<b>Prompt Template & Chat Prompt Template

In [2]:
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import SimpleJsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage


In [5]:
load_dotenv()
model = ChatGoogleGenerativeAI(model = "gemini-3.5-flash", google_api_key = os.getenv("GOOGLE_API_KEY"), temperature =0.5)

#### A simple string based prompt formatting

In [ ]:
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)
prompt = prompt_template.format(adjective="sad", content="British Weather")

response = model.invoke(prompt)
print(response)

content=[{'type': 'text', 'text': 'A little boy in England looks out the window at the endless, heavy grey rain and asks his grandfather, "Grandpa, does the sun ever come out?"\n\nThe old man looks wistfully into the distance, a tear forming in his eye, and says, "Yes, my boy. I remember it well. It was a Tuesday. I think it was 1984."\n\nThe boy asks, "What was it like?"\n\nThe grandfather sighs, "I don\'t know, son. I blinked and missed it, and I\'ve spent the last forty years hoping it would happen again."', 'extras': {'signature': 'ErUqCrIqAQw51scguHGTPnQyuBKvVfaD/bCC+g7Zu0XIUrhw+nt+tWnmgs7ekELpftA07dQx6R+irZWMEEUW3xFcsYOZKXqly0tPpEZdJWgXQmSQRlyZ+ChSEhjWlbjdIE/3C1g+HhQ5JuwTKySYdv4pdSFepf0OZLhM47LLP7KwGy+7UdkNvo6J7VPwoBdjldZJZYrUB/TEjjArnpblm8PJBcMx+ZMhJBGjEWF2NbmbwYmemAm4OAd4P+0o1QuMSi2O2e21HGvnRPj/PVbvgdt1RzrxSeWgNdnSuELI2d/VIeFwL4PsJqe60sc2qjaKXJjBmp9Y5u9DivWxs0AmRIovffOclpxu5Od3o883Y4vjjddjQT3Eghjtbs9OA4luwWpF80cST7KngbD5hZd7PGZdKhW9FpI6makD3WYtPAAJNqx6RbSYkxNy6OuSdYOr+hSfr6sLm7

#### ChatPromptTemplate: The prompt to chat models is a list of chat messages

Each chat message is associated with content, and an additional parameter called role. For example in the openai chat completions API, a chat message can be associated with an AI assistant, a human or a system role

In [7]:
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system" , "You are a helpful AI bot. Your name is {name}"),
        ("human", "Hello, How are you doing?"),
        ("ai","I am doing well, thanks!"),
        ("human","{user_input}")
    ]
)
prompt = chat_template.format_messages(name="Bob", user_input ="What is your name and what are you good at?")
respone = model.invoke(prompt)
print(response)


#### Various ways of formatting system/Human/AI prompts

In [ ]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content = (
                "You are helpful assistant that re-writes the user's text to "
                "sound more upbeat."
            )
        ),
        HumanMessagePromptTemplate.from_template("{text}"),
    ]
)

prompt = chat_template.format_messages(text="I don't like easting tastiy things")

response = model.invoke(prompt)
print(response)

#### Providing a context along with the prompt

In [ ]:
prompt = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Large Language Models (LLMs) are the latest models used in NLP.
Their superior performance over smaller models has made them incredibly
useful for developers building NLP enabled applications. These models
can be accessed via Hugging Face's `transformers` library, via OpenAI
using the `openai` library, and via Cohere using the `cohere` library.

Question: Which libraries and model providers offer LLMs?

Answer: """

In [ ]:
print(model.invoke(prompt))

#### Langchain FEWSHOTPROMPTTEMPLATE caters to source knowledge input. The idea is to train the model on a few example -- we call this few-shot-learning -- ad these examples are given to the model within the prompt

The goal of few-shot prompt template are to dynamically select examples based on an input, and then format the examples in a final prompt to provide for the model.

<b>Fixed Examples</b><br>
The most basic few shot prompting texhnique is to use a fixed prompt example. This way we can select a chain, evaluate it, and avoid worrying about additional moving parts in production.<Br>

The basic component of the template are: -example: A list of dictionary examples to include in the final prompt. example_prompt: converts each example into 1 or more messages throgh its format_messages method. A common example would be to convert each example into one human message and one AI message response, r a human message followed by a function call message.


In [9]:
from langchain_classic import FewShotPromptTemplate

# create our examples
examples = [
    {
        "query":"How are you?",
        "answer":"I can't complain but sometimes i still do"
    },
    {
        "query":"What time is it?",
        "answer":"It's time to get a watch."
    }
]

# create a example template
example_template ="""
user: {query}
Ai :{answer}
"""



In [10]:
# create a prompt example from above template

example_prompt = PromptTemplate(
    input_variables=["query","answer"],
    template= example_template
)

# now break our previous prompt into a prefix and suffix
# the prefix is our instruction
prefix = """The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 
"""
suffix="""
User:{query}
AI:
"""

In [15]:
few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix = prefix,
    suffix = suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

In [17]:
query="What movies should i watch this evening?"
print(few_shot_prompt_template.format(query=query))

The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 



user: How are you?
Ai :I can't complain but sometimes i still do



user: What time is it?
Ai :It's time to get a watch.



User:What movies should i watch this evening?
AI:



In [19]:
chain =  few_shot_prompt_template | model
chain.invoke({"query" : query})

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}